In [1]:
ddi_ai_mechanism_prompt = """### **Guideline for Determining Clinically Relevant Drug Interactions**

A clinically relevant interaction occurs when the co-administration of two drugs leads to a significant change in the therapeutic effect or adverse effect profile of one or both drugs. To determine this, check for the following two major types of interactions: Pharmacodynamic and Pharmacokinetic.

If the answer to **any** of the questions below is "yes," the interaction is clinically relevant.

---

#### **Part 1: Pharmacodynamic Interactions (What the drugs do to the body)**

This type of interaction involves drugs acting on the same or related physiological systems.

*   **Check for Antagonism:** Do the two drugs have opposing effects?
    *   **Question:** Will one drug's mechanism of action directly counteract the desired effect of the other drug?
*   **Check for Additive or Synergistic Effects:** Do the two drugs have similar effects that combine?
    *   **Question:** Do both drugs share a significant therapeutic or adverse effect, such that their combined use would dangerously enhance this effect?

---

#### **Part 2: Pharmacokinetic Interactions (What the body does to the drugs)**

This type of interaction involves one drug affecting the Absorption, Distribution, Metabolism, or Excretion (ADME) of another.

*   **Check for Absorption Issues:** Does one drug prevent the other from being absorbed into the bloodstream?
    *   **Question:** Does one drug physically or chemically prevent the other from being absorbed?
*   **Check for Metabolism Issues (Most Common):** Does one drug alter the enzyme system (most often Cytochrome P450, or CYP enzymes) that metabolizes the other?
    *   **Inhibition:** Drug A is an **inhibitor** of an enzyme that breaks down Drug B. This will cause levels of Drug B to **increase**, leading to a higher risk of **toxicity and adverse effects**. This is especially dangerous if Drug B has a **narrow therapeutic index** (the gap between a therapeutic dose and a toxic dose is small).
    *   **Induction:** Drug A is an **inducer** of an enzyme that breaks down Drug B. This will cause levels of Drug B to **decrease**, leading to a loss of effectiveness and potential **treatment failure**. This is also critical for drugs with a narrow therapeutic index.
    *   **Question:** Is one drug a known inhibitor or inducer of a key metabolic pathway for the other drug, leading to a significant change in its concentration?

---

### **Final Decision Framework**

1.  Analyze the drug pair based on the Pharmacodynamic and Pharmacokinetic checkpoints above.
2.  If **any** of the checks result in a predictable, significant negative outcome (including reduced efficacy, treatment failure, increased toxicity, or an increased risk of a serious adverse event), the interaction is clinically relevant. End your response with ?? YES ??.
3.  If no such interaction is identified, it is not considered clinically relevant. In this case, end your response with ?? NO ??.

---

Now, please apply this guideline to the question whether there exists a clinically relevant drug‑drug‑interaction between (A) {DRUG_A} and (B) {DRUG_B}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (?? YES ?? or ?? NO ??)."""

In [2]:
import re
import pandas as pd
import csv
import os
import json

In [3]:
# ["P_E1","P_E2", "P_E5","P_M","P_E3","P_MAI","P_X","P_H"]
prompt_types = dict(P_MAI=ddi_ai_mechanism_prompt)

In [6]:
import glob
# load ground truth files
DRUG_ID_MAP = {
    "upadacitinib": "DB15091",
    "digitoxin": "DB01396",
    "simvastatin": "DB00641",
}

def read_pharma_gt_csv(x):
    gt = pd.read_csv(x)
    dataset = x.split("/")[1]
    gt["DRUG_A"] = dataset
    gt["DRUG_A_ID"] = DRUG_ID_MAP[dataset]
    gt["GT"] = gt.GT.apply(lambda x: x.lower())
    return gt

gt_csvs = glob.glob("phase1/**/*final_dataset.csv", recursive=False)


pharma_gt = pd.concat([read_pharma_gt_csv(x) for x in gt_csvs])

ID_TO_DRUGNAME_MAP = {row["DRUG_B_ID"]:row["DRUG_B_NAME"] for _, row in pharma_gt.iterrows()}
for k,v in DRUG_ID_MAP.items():
    ID_TO_DRUGNAME_MAP[v] = k.capitalize()

In [14]:
def create_prompt_dicts(row):
    dicts = []
    for prompt_type, prompt_template in prompt_types.items():
        p = prompt_template.format(DRUG_A=row.DRUG_A.capitalize(), DRUG_B=row.DRUG_B_NAME)
        cid = row.DRUG_A_ID + "__" + row.DRUG_B_ID + "__" + prompt_type + "__0"
        d = dict(custom_id=cid, method="POST",url="/v1/chat/completions", 
                 body=dict(model="X", messages=[dict(role="user",content=p)]))
        dicts.append(d)
    return dicts

requests = pharma_gt.apply(create_prompt_dicts , axis=1)

In [ ]:
requests.explode().to_json("data/ddi.jsonl", orient="records", lines=True, force_ascii=False)